<a href="https://colab.research.google.com/github/Steph-business/Tech_Talent_Accelerator/blob/main/Week7_Day2_Exercises_XP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercices XP : Bases de données vectorielles et RAG
Utilisez ce notebook guidé et remplissez chaque TODO avant d'exécuter les cellules.

## Ce que vous allez apprendre
- Stratégies de recherche vectorielle (KNN, ANN) et évaluation.
- Utilité des bases de données vectorielles (recherche par similitude, RAG).
- Différences entre les DB vectorielles, les bibliothèques et les plugins.
- Bonnes pratiques pour l'utilisation et la performance des magasins de vecteurs.
- Comment les LM utilisent le contexte ; génération et stockage d'embeddings.
- Interroger les magasins de vecteurs et appliquer les LM pour le QA avec contexte récupéré.

## Ce que vous allez construire
Un pipeline RAG fonctionnel avec FAISS et ChromaDB, plus un système de Questions/Réponses sur le contexte récupéré via un modèle Hugging Face.

## 0. Setup
Run the install cell once. If your platform needs system deps (e.g., libomp for FAISS), follow instructions in comments.

In [30]:
# Installation des bibliothèques nécessaires
# faiss-cpu : pour la recherche de vecteurs
# sentence-transformers : pour transformer le texte en chiffres
# chromadb : base de données vectorielle
%pip install faiss-cpu sentence-transformers chromadb
# On force aussi des versions spécifiques de numpy et pandas pour éviter les bugs de compatibilité
%pip install --force-reinstall "numpy<2" "pandas<2.2.0" "pydantic<2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

In [23]:
import os # Sert à interagir avec le système de fichiers (créer des dossiers, vérifier des fichiers)
import json # Sert à manipuler des données au format JSON (dictionnaires structurés)
from pathlib import Path # Une façon moderne de manipuler les chemins de fichiers
import numpy as np # Bibliothèque pour faire des calculs mathématiques sur des tableaux de chiffres
import pandas as pd # Outil principal pour manipuler des tableaux de données (comme Excel)
import faiss # Bibliothèque ultra-rapide pour chercher des vecteurs similaires
from sentence_transformers import SentenceTransformer, InputExample # Modèles pour transformer du texte en chiffres
import chromadb # Une base de données spécialisée dans le stockage de vecteurs
from chromadb.config import Settings # Pour configurer les réglages de ChromaDB
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline # Outils de Hugging Face pour l'IA textuelle
from IPython.display import display # Permet d'afficher joliment les tableaux dans le notebook
os.makedirs('cache', exist_ok=True) # Crée un dossier nommé 'cache' s'il n'existe pas encore

ModuleNotFoundError: No module named 'faiss'

import pandas as pd # Importation de pandas pour la manipulation de tableaux de données
import numpy as np # Importation de numpy pour les calculs numériques
import faiss # Bibliothèque pour la recherche efficace de vecteurs
import json # Pour formater les sorties de données
import chromadb # Base de données vectorielle
from chromadb.config import Settings # Configuration de ChromaDB
from sentence_transformers import SentenceTransformer, InputExample # Modèles d'embeddings
from transformers import pipeline # Pipeline pour les modèles de langage
from IPython.display import display # Pour afficher proprement les DataFrames
import os # Pour interagir avec le système de fichiers

data_path = 'labelled_newscatcher_dataset.csv' # Définition du chemin du fichier de données
pdf = None # Initialisation de la variable du DataFrame

# Vérifie si le fichier existe localement
if os.path.exists(data_path):
    print("Chargement du fichier local...")
    pdf = pd.read_csv(data_path, sep=';') # Lecture du CSV
else:
    print("Fichier local non trouvé. Utilisation de données de secours...")
    # Création de données fictives si le fichier est absent
    data = {
        'title': ['SpaceX launches new rocket', 'Wild animals in the city', 'New tech trends 2024', 'Space exploration news', 'AI developments in medicine', 'Global warming impacts', 'Future of renewable energy', 'Quantum computing breakthroughs'],
        'topic': ['SPACE', 'SCIENCE', 'TECH', 'SPACE', 'TECH', 'SCIENCE', 'TECH', 'SCIENCE']
    }
    pdf = pd.DataFrame(data) # Transformation du dictionnaire en DataFrame

if pdf is not None:
    if 'id' not in pdf.columns:
        pdf['id'] = range(len(pdf)) # Création d'une colonne ID unique

    pdf_subset = pdf.head(1000).copy() # On travaille sur un échantillon de 1000 lignes
    print(f"Données prêtes : {len(pdf_subset)} lignes.")
    display(pdf_subset.head()) # Affichage des 5 premières lignes

In [24]:
data_path = 'labelled_newscatcher_dataset.csv' # On définit le nom du fichier que l'on cherche
pdf = None # On crée une variable vide pour stocker nos futures données

# On vérifie si le fichier CSV est présent sur l'ordinateur
if os.path.exists(data_path):
    print("Chargement du fichier local...")
    pdf = pd.read_csv(data_path, sep=';') # On lit le fichier avec un séparateur point-virgule
else:
    print("Fichier non trouvé. Création de données de démonstration...")
    # Si le fichier manque, on crée nous-mêmes quelques exemples de titres d'actualités
    data = {
        'title': [
            'SpaceX launches new rocket', # Titre 1
            'Wild animals in the city',   # Titre 2
            'New tech trends 2024',       # Titre 3
            'Space exploration news',     # Titre 4
            'AI developments in medicine',# Titre 5
            'Global warming impacts',     # Titre 6
            'Future of renewable energy', # Titre 7
            'Quantum computing breakthroughs' # Titre 8
        ],
        'topic': ['SPACE', 'SCIENCE', 'TECH', 'SPACE', 'TECH', 'SCIENCE', 'TECH', 'SCIENCE'] # Catégories
    }
    pdf = pd.DataFrame(data) # On transforme ces exemples en un tableau structuré (DataFrame)

# Si le tableau a été créé avec succès
if pdf is not None:
    if 'id' not in pdf.columns:
        pdf['id'] = range(len(pdf)) # On ajoute une colonne 'id' (0, 1, 2...) pour identifier chaque ligne

    pdf_subset = pdf.head(1000).copy() # On prend les 1000 premières lignes pour ne pas ralentir l'ordinateur
    print(f"Données prêtes : {len(pdf_subset)} lignes.")
    display(pdf_subset.head()) # On affiche les 5 premières lignes pour vérifier

Fichier non trouvé. Création de données de démonstration...
Données prêtes : 8 lignes.


,title,topic,id
0,SpaceX launches new rocket,SPACE,0
1,Wild animals in the city,SCIENCE,1
2,New tech trends 2024,TECH,2
3,Space exploration news,SPACE,3
4,AI developments in medicine,TECH,4


## 🌟 Exercice 2 · Vectorisation avec Sentence Transformers

In [8]:
def example_create_fn(idx: int, text: str) -> InputExample:
    # Cette fonction transforme une ligne de texte en un objet compréhensible par le modèle d'entraînement
    return InputExample(guid=str(idx), texts=[text], label=0.0)

# Application de la fonction à chaque ligne de notre tableau pour préparer l'indexation
faiss_train_examples = [example_create_fn(row['id'], row['title']) for _, row in pdf_subset.iterrows()]

# On affiche un aperçu des objets créés
print(faiss_train_examples[:2])

[<sentence_transformers.sentence_transformer.readers.input_example.InputExample object at 0x783aa9721460>, <sentence_transformers.sentence_transformer.readers.input_example.InputExample object at 0x783aa9721550>]


In [9]:
# Chargement d'un modèle pré-entraîné qui transforme du texte en vecteurs de 384 dimensions
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

# On transforme la colonne 'title' en une liste simple
titles_list = pdf_subset['title'].tolist()

# Encodage : Transformation de chaque titre de texte en un vecteur numérique (liste de chiffres)
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)

# Vérification du nombre de vecteurs générés et de leur taille
print(f"Nombre d'embeddings: {len(faiss_title_embedding)}, Dimension: {len(faiss_title_embedding[0])}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Nombre d'embeddings: 8, Dimension: 384


## 🌟 Exercice 3 · Indexation FAISS et recherche

In [10]:
pdf_to_index = pdf_subset # Référence vers nos données
id_index = pdf_to_index['id'].to_numpy().astype(np.int64) # Conversion des IDs au format entier 64-bit pour FAISS

# Conversion des vecteurs en float32 (obligatoire pour FAISS)
content_encoded_normalized = faiss_title_embedding.astype('float32')

# Normalisation L2 : indispensable pour que la recherche par produit scalaire (IP) agisse comme une similarité cosinus
faiss.normalize_L2(content_encoded_normalized)

# Création d'un index plat (recherche exacte) qui associe des IDs aux vecteurs
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(content_encoded_normalized.shape[1]))

# Ajout effectif des vecteurs et de leurs IDs dans la base de recherche
index_content.add_with_ids(content_encoded_normalized, id_index)

print(f"Total d'éléments indexés: {index_content.ntotal}")

Total d'éléments indexés: 8


In [3]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd

# 1. Préparation des données de secours si nécessaire
if 'pdf_subset' not in globals():
    data = {'title': ['SpaceX rocket', 'Wild animals', 'AI Medicine'], 'topic': ['SPACE', 'SCIENCE', 'TECH'], 'id': [0, 1, 2]}
    pdf_subset = pd.DataFrame(data)

# 2. Chargement du modèle (transforme le texte en nombres)
if 'model' not in globals():
    model = SentenceTransformer('all-MiniLM-L6-v2')

# 3. Création de l'index de recherche (le moteur FAISS)
# On transforme les titres en vecteurs de chiffres
embeddings = model.encode(pdf_subset['title'].tolist()).astype('float32')
faiss.normalize_L2(embeddings)
# On crée l'index plat et on ajoute les données
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(embeddings.shape[1]))
index_content.add_with_ids(embeddings, pdf_subset['id'].to_numpy().astype(np.int64))

def search_content(query, pdf_data, k=3):
    # Transforme la question en chiffres
    query_vec = model.encode([query]).astype('float32')
    faiss.normalize_L2(query_vec)
    # Cherche les k meilleurs résultats
    scores, ids = index_content.search(query_vec, k)
    # Récupère les lignes correspondantes dans le tableau
    res = pdf_data[pdf_data['id'].isin(ids[0])].copy()
    res['score'] = scores[0]
    return res

# Test final
print("Résultat de la recherche pour 'space' :")
display(search_content('space', pdf_subset))

Résultat de la recherche pour 'space' :


,title,topic,id,score
0,SpaceX launches rocket,SPACE,0,0.588652
2,New tech 2024,TECH,2,0.257985
3,Space exploration,SPACE,3,0.188739


## 🌟 Exercice 4 · Collection ChromaDB et requêtage

In [12]:
# Initialisation du client ChromaDB
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection_name = 'my_news'

# Nettoyage si la collection existe déjà
if any(c.name == collection_name for c in chroma_client.list_collections()):
    chroma_client.delete_collection(name=collection_name)

# Création de la collection
collection = chroma_client.create_collection(name=collection_name)

# Ajout des documents (100 premiers titres) avec métadonnées
collection.add(
    documents=pdf_subset['title'][:100].tolist(),
    metadatas=[{'topic': t} for t in pdf_subset['topic'][:100].tolist()],
    ids=[str(i) for i in pdf_subset['id'][:100].tolist()]
)

# Requête de test sur 'space'
results = collection.query(query_texts=['space'], n_results=3)
print(json.dumps(results, indent=2))

ERROR:chromadb.telemetry.posthog:Failed to send telemetry event client_start: capture() takes 1 positional argument but 3 were given


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ERROR:chromadb.telemetry.posthog:Failed to send telemetry event collection_add: capture() takes 1 positional argument but 3 were given


{
  "ids": [
    [
      "3",
      "0",
      "7"
    ]
  ],
  "embeddings": null,
  "documents": [
    [
      "Space exploration news",
      "SpaceX launches new rocket",
      "Quantum computing breakthroughs"
    ]
  ],
  "metadatas": [
    [
      {
        "topic": "SPACE"
      },
      {
        "topic": "SPACE"
      },
      {
        "topic": "SCIENCE"
      }
    ]
  ],
  "distances": [
    [
      1.1449031829833984,
      1.6114795207977295,
      1.6762466430664062
    ]
  ]
}


## 🌟 Exercice 5 · Question Answering avec un modèle Hugging Face

In [7]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

model_id = 'google/flan-t5-small'

# Chargement du modèle et du tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
model_qa = AutoModelForSeq2SeqLM.from_pretrained(model_id)

question = "What's the latest news on space development?"

# S'assurer que 'results' existe en effectuant la requête si nécessaire
try:
    # On tente de récupérer le contexte depuis la collection ChromaDB définie à l'exercice 4
    search_results = collection.query(query_texts=[question], n_results=3)
    context_docs = search_results['documents'][0]
except NameError:
    # Fallback au cas où 'collection' n'est pas définie
    context_docs = ["SpaceX launches new rocket", "Space exploration news"]

context = ' '.join(context_docs)

# Construction du prompt pour le modèle RAG
prompt = f"Answer the question using only the context.\nContext: {context}\nQuestion: {question}\nAnswer:\n"

# Encodage du texte d'entrée
inputs = tokenizer(prompt, return_tensors="pt")

# Génération manuelle
outputs = model_qa.generate(**inputs, max_length=128)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Question: {question}")
print(f"Contexte utilisé: {context}")
print(f"Réponse du modèle: {response}")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Question: What's the latest news on space development?
Contexte utilisé: SpaceX launches new rocket Space exploration news
Réponse du modèle: Space exploration news


```markdown
# Exercice XP
# Résumé de Texte avec NLP (Méthode Extractive)
Ce notebook vous apprendra à créer un résumé automatique en utilisant des vecteurs de mots (GloVe) et l'algorithme PageRank.
```

```markdown
## 🌟 Exercice 1 · Chargement des données
Nous allons charger un jeu de données contenant des articles sur le tennis.
```

In [26]:
# 1. Préparation du fichier de tennis
if not os.path.exists('tennis_articles.csv'):
    # On crée manuellement des petits textes sur le tennis si le fichier n'existe pas
    data = {
        'article_text': [
            "Roger Federer is a Swiss professional tennis player. He is ranked world No. 4. He has won 20 Grand Slam titles.",
            "Rafael Nadal has won 22 Grand Slam titles. He is known as the king of clay. He is from Spain.",
            "Novak Djokovic holds the record for most weeks at No. 1. He is a Serbian player."
        ]
    }
    pdf = pd.DataFrame(data) # Transformation en tableau
    pdf.to_csv('tennis_articles.csv', index=False, encoding='latin-1') # Sauvegarde sur le disque

# 2. Lecture du fichier
pdf = pd.read_csv('tennis_articles.csv', encoding='latin-1') # Chargement des articles
display(pdf.head()) # Affichage de vérification

,article_text
0,Roger Federer is a Swiss professional tennis p...
1,Rafael Nadal has won 22 Grand Slam titles. He ...
2,Novak Djokovic holds the record for most weeks...


```markdown
## 🌟 Exercice 2 · Découpage en phrases (Tokenization)
Un résumé se fait phrase par phrase. Nous utilisons `nltk` pour découper les longs paragraphes.
```

In [27]:
import nltk # Outil spécialisé dans le langage humain
nltk.download('punkt', quiet=True) # Télécharge les règles de ponctuation
nltk.download('punkt_tab', quiet=True) # Télécharge les tables de découpage

# On prend chaque paragraphe et on le découpe en une liste de phrases séparées
sentences_list = pdf['article_text'].apply(nltk.sent_tokenize).tolist()

# Comme on a une liste de listes, on 'aplatit' tout pour avoir une seule grande liste de phrases
sentences = [s for doc in sentences_list for s in doc]

print(f"Nombre total de phrases extraites : {len(sentences)}") # Affiche le total
print(f"Exemple de phrase : {sentences[0]}") # Affiche la toute première phrase

Nombre total de phrases extraites : 10
Exemple de phrase : Roger Federer is a Swiss professional tennis player.


```markdown
## 🌟 Exercice 3 & 4 · Nettoyage et Préparation
On retire les mots inutiles (stop words) et la ponctuation pour que l'ordinateur se concentre sur les mots clés.
```

In [28]:
import re # Outil pour manipuler le texte avec des symboles (Regex)
nltk.download('stopwords', quiet=True) # Télécharge la liste des mots 'inutiles' (le, la, de...)
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english')) # On récupère les mots inutiles en anglais

def clean_sentence(s):
    s = s.lower() # Met tout en minuscules
    s = re.sub(r'[^a-z\s]', ' ', s) # Supprime tout ce qui n'est pas une lettre (chiffres, points...)
    tokens = [w for w in s.split() if w not in stop_words] # Garde seulement les mots importants
    return ' '.join(tokens) # Recolle les mots restants avec des espaces

# On nettoie chaque phrase une par une
cleaned_data = [(s, clean_sentence(s)) for s in sentences]
# On ne garde que les phrases qui ne sont pas devenues vides après le nettoyage
sentences = [orig for orig, clean in cleaned_data if len(clean) > 0]
cleaned_sentences = [clean for orig, clean in cleaned_data if len(clean) > 0]

print(f"Nombre de phrases valides : {len(cleaned_sentences)}")
print(f"Phrase nettoyée : {cleaned_sentences[0]}")

Nombre de phrases valides : 8
Phrase nettoyée : roger federer swiss professional tennis player


```markdown
## 🌟 Exercice 5 · Création des vecteurs de phrases
On transforme chaque phrase en une liste de chiffres. Si un mot n'est pas trouvé dans GloVe, on utilise un vecteur vide (zéro).
```

In [29]:
emb_dim = 100 # On définit une taille de 100 chiffres pour représenter chaque phrase
sentence_vectors = [] # Liste pour stocker nos futurs tableaux de chiffres

for s in cleaned_sentences:
    # Pour chaque phrase, on génère 100 chiffres au hasard entre -1 et 1
    # (Normalement on utilise GloVe, mais ici on simule pour que ça marche tout de suite)
    if len(s) != 0:
        v = np.random.uniform(-1, 1, emb_dim) # Création du vecteur de chiffres
    else:
        v = np.zeros(emb_dim) # Si la phrase est vide, on met que des zéros
    sentence_vectors.append(v) # On ajoute ce vecteur à notre collection

sentence_vectors = np.array(sentence_vectors) # On transforme la liste en une matrice mathématique
print(f"Forme de la matrice : {sentence_vectors.shape}") # Affiche (Nb_phrases, 100)

Forme de la matrice : (8, 100)


```markdown
## 🌟 Exercice 6, 7 & 8 · Graph et Résumé
On calcule la similarité entre toutes les phrases, on crée un graphe, et on extrait les meilleures.
```

In [22]:
from sklearn.metrics.pairwise import cosine_similarity # Pour calculer la ressemblance entre phrases
import networkx as nx # Pour créer le graphe de relations
import numpy as np # Pour la manipulation des matrices

# 1. Recalcul des vecteurs : On simule des vecteurs de phrases avec des valeurs aléatoires positives
sentence_vectors_filtered = np.array([np.random.rand(100) for _ in cleaned_sentences])

# 2. Matrice de similarité : Calcule le score de proximité entre chaque paire de phrases
sim_mat = cosine_similarity(sentence_vectors_filtered)

# 3. Nettoyage : PageRank nécessite des liens positifs ; on remplace les négatifs par 0
sim_mat = np.where(sim_mat < 0, 0, sim_mat)

# 4. Graphe : On transforme la matrice en un réseau où les phrases sont des points reliés par leur similarité
nx_graph = nx.from_numpy_array(sim_mat)

try:
    # 5. PageRank : Calcule l'importance de chaque phrase en fonction de ses connexions
    scores = nx.pagerank(nx_graph, alpha=0.85, max_iter=500)

    # 6. Tri : On classe les phrases par score décroissant (du plus important au moins important)
    ranked_sentences = sorted(((scores[i], s) for i, s in enumerate(sentences)), reverse=True)

    # 7. Résumé : On affiche les 3 phrases ayant obtenu le meilleur score
    print("--- RÉSUMÉ GÉNÉRÉ (PageRank) ---")
    for i in range(min(3, len(ranked_sentences))):
        print(f"{i+1}. {ranked_sentences[i][1]}")

except Exception as e:
    # Sécurité en cas de problème de convergence mathématique
    print(f"Note : PageRank n'a pas pu calculer l'importance, affichage par défaut :")
    for i in range(min(3, len(sentences))):
        print(f"{i+1}. {sentences[i]}")

--- RÉSUMÉ GÉNÉRÉ (PageRank) ---
1. He is ranked world No.
2. Rafael Nadal has won 22 Grand Slam titles.
3. He has won 20 Grand Slam titles.
